# Best filter for envelope extraction

Previous notebooke titled "[filter search for envelope extraction](3_filter_search_for_envelope_extraction.ipynb)" introduced a method for filter search for envelope extraction and its visualization. This notebook builds upon it. In cases where multiple measurements of simmilar fault exists, automatic methods to extract best filter are desirable.

First, let's examine measurements from different electric motors with faulty bearings. All motors contain bearings with similar types of faults. This is important because cross-validation is used when searching for the best filter: one measurement is held out during filter selection, and the selected filter is then evaluated on this held-out measurement.

We already know that different bearing faults can manifest at different bearing-fault characteristic frequencies. Therefore, if cross-validation were performed across measurements containing completely different fault types, the selected filter might be optimized for the characteristics of one fault while being unsuitable for another, potentially resulting in poor performance on the held-out measurement.

In [1]:
import pickle

with open("data/em_measurements.pkl", "rb") as f:
    data = pickle.load(f)

t = data["t"]
sp_1 = data["sp_1"]
sp_2 = data["sp_2"]
sp_3 = data["sp_3"]
sp_4 = data["sp_4"]

First `filter_search_for_envelope_extraction` is used on all different measurements, to obtain multiple possible candidates for best filter to extract envelope.

In [2]:
from pybearing.core.bearing import Bearing
from pybearing.analysis.fault_frequencies_analysis import get_fault_frequencies
from pybearing.analysis.envelope_analysis import filter_search_for_envelope_extraction

bearing = Bearing(name = "NMB_1560kk", d = 6, D = 15, B = 5)

fault_frequencies = get_fault_frequencies(bearing, 2500, 1, method="database")

results = []
for sp in [sp_1, sp_2, sp_3, sp_4]:
    result = filter_search_for_envelope_extraction(
        x=sp, 
        fs=50000,
        fault_frequencies=fault_frequencies, 
        max_level=6,
        compute_triadic=True, 
        max_frequency=12000,
        min_frequency=20,
        epsilon=1e-4,
        calculate_harmonics_score=True,
        n_harmonics=3,
        min_samples=2, 
        absolute_error=0.5,
        relative_error=0.04,
        calculate_kurtosis_score=False
    )
    results.append(result)

Then, the list of `results` is passed to the `best_filter_for_envelope_extraction` method. Currently, search for best filter is only implemented on harmonics scores. Objective function for determining the best filter for envelope extraction is calculated on z-normalised scores for each measurement. Z-normalisation is performed so that it is possible to compare scores from different measurements. Default objective function is calcultaed as $mean(score) - 0.5 \cdot std(score)$, therefore penalising high deviations between scores of the same band. For valuation of bands cross-validation is implemented.

In [ ]:
from pybearing.analysis.envelope_analysis import best_filter_for_envelope_extraction

results = best_filter_for_envelope_extraction(
    x=results,
    signal_names=["sp_1", "sp_2", "sp_3", "sp_4"],
    evaluate_harmonics_score=True
)

There are several results, that we can observe. First `summary_harmonics_score`, provides us with all possible selected bands. If we take a look at fault **cage**, there are two different bands. The first band from 20 Hz to 270 Hz was selected as the best band in 3 out of 4 cases (4 case as there are 4 measurements and due to cross-validation each of the four measurements is held out one time while the best band is searched for on the remaigning 3 measurements). We are looking for a frequency band with high **mean_test_score**, low **std_test_score** and low **mean_rank**. For example in case of **inner_ring** frequency band from 2017 Hz to 4013 Hz was selected in all 4 cases as the best band. With a **mean_rank** of 1.25 and **max_rank** of 2. **max_rank** provides the information that on held out measurement selected band on other three bands was the second best band for heald out measurement. In cases of **rolling_element** and **rolling_element_about_axis**, we can see multiple bands with high **max_rank** values, indicating that analysed measurements of bearings are not having damaged rolling elements, which is correct for the provided measurements. This is also the reason that all measurements of the bearings should contain simmilar damage, otherwise determining best band for each fault could be imposible.

In [5]:
results["summary_harmonics_score"]

,fault,level,f_low,f_high,mean_test_score,std_test_score,mean_rank,max_rank,n,selection_count,selection_frequency
0,cage,4.585,20.000000,269.583333,4.869831,0.245649,2.333333,3,3,3,0.75
1,cage,5.000,20.000000,394.375000,4.709676,NaN,2.000000,2,1,1,0.25
2,inner_ring,1.585,2016.666667,4013.333333,7.281633,0.285593,1.250000,2,4,4,1.00
3,outer_ring,2.585,2016.666667,3015.000000,6.114827,0.287201,2.500000,3,2,2,0.50
4,outer_ring,3.585,2016.666667,2515.833333,3.688511,2.047708,8.500000,15,2,2,0.50
7,rolling_element,3.000,1517.500000,3015.000000,3.050529,0.331452,12.000000,16,2,2,0.50
6,rolling_element,1.585,2016.666667,4013.333333,2.585113,NaN,10.000000,10,1,1,0.25
5,rolling_element,0.000,20.000000,12000.000000,1.814716,NaN,13.000000,13,1,1,0.25
8,rolling_element_about_axis,0.000,20.000000,12000.000000,3.648399,0.384442,10.000000,11,3,3,0.75
9,rolling_element_about_axis,4.000,20.000000,768.750000,1.255085,NaN,29.000000,29,1,1,0.25


Secondly `selection_results_harmonics_score` provides whole information of the calculation prcess. It describes results of the cross-validation. Explanation of the first row of the dataframe:
1. signal: "sp_1" name of the held out measurement.
2. fault: "cage" performing calculation on harmonics scores of the cage characteristic frequency and its sidebands.
3. level: 4.585 level of the filter (used for plotting).
4. f_low: 20 Hz lower frequency of the filter
5. f_high: 270 Hz higher frequency of the filter
6. test_score: 4.59 score if we extract envelope with filter from 20 Hz to 270 Hz and calculate harmonics score.
7. rank: 3, filter from 20 Hz to 270 Hz was determined on signals "sp_2", "sp_3" and "sp_4". If using this filter we get test_score of 4.59, which is the 3rd highest possible score on signal "sp_1".

In [6]:
results["selection_results_harmonics_score"]

,signal,fault,level,f_low,f_high,test_score,rank
0,sp_1,cage,4.585,20.000000,269.583333,4.588697,3
1,sp_2,cage,5.000,20.000000,394.375000,4.709676,2
2,sp_3,cage,4.585,20.000000,269.583333,5.043049,2
3,sp_4,cage,4.585,20.000000,269.583333,4.977747,2
4,sp_1,rolling_element_about_axis,0.000,20.000000,12000.000000,4.051177,10
5,sp_2,rolling_element_about_axis,0.000,20.000000,12000.000000,3.608637,11
6,sp_3,rolling_element_about_axis,0.000,20.000000,12000.000000,3.285383,9
7,sp_4,rolling_element_about_axis,4.000,20.000000,768.750000,1.255085,29
8,sp_1,outer_ring,3.585,2016.666667,2515.833333,2.240563,15
9,sp_2,outer_ring,2.585,2016.666667,3015.000000,6.317909,3
